# STC Jawwy

In [ ]:
"""
Here we install libraries that are not installed by default
Example:  pyslsb
Feel free to add any library you are planning to use.
"""
!pip install pyxlsb

In [ ]:
# Import the required libraries
"""
Please feel free to import any required libraries as per your needs
"""
import pandas as pd     # provides high-performance, easy to use structures and data analysis tools
import pyxlsb           # Excel extention to read xlsb files (the input file)
import numpy as np      # provides fast mathematical computation on arrays and matrices

# Jawwy dataset
The dataset includes total watching hours for customers per day.

You are required to work on predecting the forecast for the watching hours.

In [ ]:
dataframe = pd.read_excel("/content/sample_data/stc TV Data Set_T2.xlsx",index_col=0)
# Please make a copy of dataset if you are going to work directly and make changes on the dataset
# you can use   df=dataframe.copy()

In [ ]:
# check the data shape
dataframe.shape

(86, 2)

In [ ]:
# display the first 5 rows
dataframe.head()

,date_,Total_watch_time_in_houres
0,2018-01-01,1123.551944
1,2018-01-02,1000.129722
2,2018-01-03,881.924444
3,2018-01-04,782.669444
4,2018-01-05,1051.939444


In [ ]:
# display the dataset after applying data types
dataframe.head()

,date_,Total_watch_time_in_houres
0,2018-01-01,1123.551944
1,2018-01-02,1000.129722
2,2018-01-03,881.924444
3,2018-01-04,782.669444
4,2018-01-05,1051.939444


In [ ]:
# describe the numeric values in the dataset
dataframe.describe()

,date_,Total_watch_time_in_houres
count,86,86.000000
mean,2018-02-28 17:01:23.720930304,780.817926
min,2018-01-01 00:00:00,562.124722
25%,2018-01-30 06:00:00,707.709653
50%,2018-02-28 12:00:00,763.181389
75%,2018-03-29 18:00:00,840.985278
max,2018-04-30 00:00:00,1123.551944
std,NaN,122.992002


In [ ]:
# check if any column has null value in the dataset
dataframe.isnull().any()

,0
date_,False
Total_watch_time_in_houres,False


In [ ]:
# we import Visualization libraries
# you can ignore and use any other graphing libraries
import matplotlib.pyplot as plt # a comprehensive library for creating static, animated, and interactive visualizations
import plotly #a graphing library makes interactive, publication-quality graphs. Examples of how to make line plots, scatter plots, area charts, bar charts, error bars, box plots, histograms, heatmaps, subplots, multiple-axes, polar charts, and bubble charts.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Setting the date as index
dataframe.set_index('date_', inplace=True)

In [ ]:
# Display the dataframe after setting the date as index
dataframe.head()

,Total_watch_time_in_houres
date_,
2018-01-01,1123.551944
2018-01-02,1000.129722
2018-01-03,881.924444
2018-01-04,782.669444
2018-01-05,1051.939444


In [ ]:
!pip install statsmodels

In [ ]:
# Time series analysis for forecasting
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Split data into train and test sets
train_size = int(len(dataframe) * 0.8)
train, test = dataframe.iloc[:train_size], dataframe.iloc[train_size:]

print(f"Training data: {len(train)} days")
print(f"Testing data: {len(test)} days")

Training data: 68 days
Testing data: 18 days


In [ ]:
# Holt-Winters model for seasonal forecasting
model = ExponentialSmoothing(
    train['Total_watch_time_in_houres'],
    seasonal_periods=7,  # Weekly seasonality
    trend='add',
    seasonal='add'
)
fitted_model = model.fit()
print("Model training completed!")

Model training completed!


In [ ]:
# Forecast for next 60 days
forecast_days = 60
forecast = fitted_model.forecast(forecast_days)

# Create future dates
last_date = dataframe.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_days)

# Display forecast results
forecast_df = pd.DataFrame({
    'date': future_dates,
    'predicted_watch_time': forecast.values
})
print("First 10 days of forecast:")
print(forecast_df.head(10))

First 10 days of forecast:
        date  predicted_watch_time
0 2018-05-01            707.834059
1 2018-05-02            669.802909
2 2018-05-03            689.544537
3 2018-05-04            697.529844
4 2018-05-05            692.820003
5 2018-05-06            671.794203
6 2018-05-07            697.573387
7 2018-05-08            682.197302
8 2018-05-09            644.166153
9 2018-05-10            663.907781


In [ ]:
# Predict on test data
test_predictions = fitted_model.forecast(len(test))

# Calculate metrics
mae = mean_absolute_error(test['Total_watch_time_in_houres'], test_predictions)
rmse = np.sqrt(mean_squared_error(test['Total_watch_time_in_houres'], test_predictions))
mape = np.mean(np.abs((test['Total_watch_time_in_houres'].values - test_predictions) / test['Total_watch_time_in_houres'].values)) * 100

print(f"MAE (Mean Absolute Error): {mae:.2f}")
print(f"RMSE (Root Mean Square Error): {rmse:.2f}")
print(f"MAPE (Mean Absolute Percentage Error): {mape:.2f}%")

MAE (Mean Absolute Error): 57.17
RMSE (Root Mean Square Error): 66.73
MAPE (Mean Absolute Percentage Error): 8.60%


In [ ]:
# Identify peak days in historical data
peak_threshold = dataframe['Total_watch_time_in_houres'].quantile(0.85)
peak_days = dataframe[dataframe['Total_watch_time_in_houres'] > peak_threshold]

print(f"Number of peak days: {len(peak_days)}")
print("\nTop 10 peak days:")
print(peak_days.sort_values('Total_watch_time_in_houres', ascending=False).head(10))

# Day of week analysis
dataframe['weekday'] = dataframe.index.dayofweek
weekday_avg = dataframe.groupby('weekday')['Total_watch_time_in_houres'].mean()
weekday_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

print("\nAverage watch time by day of week:")
for i, avg in enumerate(weekday_avg):
    print(f"  {weekday_names[i]}: {avg:.2f} hours")

Number of peak days: 13

Top 10 peak days:
            Total_watch_time_in_houres
date_                                 
2018-01-01                 1123.551944
2018-02-01                 1053.168611
2018-01-05                 1051.939444
2018-02-06                 1025.595556
2018-01-22                 1016.186667
2018-02-05                 1012.082222
2018-01-02                 1000.129722
2018-03-22                  981.626944
2018-01-10                  970.475000
2018-02-09                  950.520000

Average watch time by day of week:
  Monday: 790.17 hours
  Tuesday: 785.23 hours
  Wednesday: 766.28 hours
  Thursday: 767.93 hours
  Friday: 793.92 hours


In [ ]:
# Plot historical data and forecast
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dataframe.index,
    y=dataframe['Total_watch_time_in_houres'],
    mode='lines',
    name='Historical Data'
))
fig.add_trace(go.Scatter(
    x=future_dates,
    y=forecast,
    mode='lines',
    name='Forecast (2 months)',
    line=dict(dash='dash')
))

fig.update_layout(
    title='Watch Time Forecast for Next 60 Days',
    xaxis_title='Date',
    yaxis_title='Watch Time (hours)',
    legend_title='Data Type'
)
fig.show()

In [ ]:
# show the dataframe
fig = px.line(dataframe,  y="Total_watch_time_in_houres")
fig.show()

In [ ]:
"""
TODO using the previous dataset (df) build a prediction model to predict the expected watch time for the next two months
Hint: you can build a forecast model to predict the results
"""

'\nTODO using the previous dataset (df) build a prediction model to predict the expected watch time for the next two months\nHint: you can build a forecast model to predict the results\n'

In [ ]:
print("=" * 60)
print("تحليل اتجاه المشاهدات - هل تتوقع ارتفاع؟")
print("=" * 60)

# Calculate monthly averages
dataframe['month'] = dataframe.index.month
monthly_avg = dataframe.groupby('month')['Total_watch_time_in_houres'].mean()

print("\n📊 متوسط المشاهدات الشهري:")
print(f"   يناير (الشهر 1): {monthly_avg[1]:.2f} ساعة")
if 2 in monthly_avg.index:
    print(f"   فبراير (الشهر 2): {monthly_avg[2]:.2f} ساعة")
if 3 in monthly_avg.index:
    print(f"   مارس (الشهر 3): {monthly_avg[3]:.2f} ساعة")

# Check trend
if 3 in monthly_avg.index and 1 in monthly_avg.index:
    growth = ((monthly_avg[3] - monthly_avg[1]) / monthly_avg[1]) * 100
    print(f"\n📈 نسبة النمو من يناير إلى مارس: {growth:.1f}%")

    if growth > 0:
        print("   ✅ النتيجة: المشاهدات في ارتفاع")
    else:
        print("   ❌ النتيجة: المشاهدات في انخفاض")

# Check overall trend using linear regression
from scipy import stats

x = range(len(dataframe))
y = dataframe['Total_watch_time_in_houres'].values
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

print(f"\n📐 تحليل الاتجاه الخطي:")
print(f"   قيمة الميل (Slope): {slope:.4f}")

if slope > 0:
    print("   ✅ الميل موجب → المشاهدات في ارتفاع")
    print(f"   ⬆️ متوسط الزيادة اليومية: {slope:.2f} ساعة")
else:
    print("   ❌ الميل سالب → المشاهدات في انخفاض")

# Forecast future trend using your model
forecast_days = 60
forecast = fitted_model.forecast(forecast_days)

last_30_avg = dataframe['Total_watch_time_in_houres'].tail(30).mean()
next_30_avg = forecast[:30].mean()

print(f"\n🔮 مقارنة آخر 30 يوم بالـ 30 يوم القادمة:")
print(f"   آخر 30 يوم (فعلي): {last_30_avg:.2f} ساعة")
print(f"   الـ 30 يوم القادمة (متوقع): {next_30_avg:.2f} ساعة")

if next_30_avg > last_30_avg:
    print("   ✅ المتوقع أعلى → استمرار الارتفاع")
    increase_pct = ((next_30_avg - last_30_avg) / last_30_avg) * 100
    print(f"   ⬆️ نسبة الزيادة المتوقعة: {increase_pct:.1f}%")
else:
    print("   ⚠️ المتوقع أقل أو مساوٍ")

# Final answer
print("\n" + "=" * 60)
print("🏆 الإجابة النهائية:")
print("=" * 60)

if slope > 0 and (next_30_avg > last_30_avg or growth > 0):
    print("✅ صحيح - من المتوقع أن ترتفع المشاهدات في stc tv")
else:
    print("❌ غير صحيح - المشاهدات في انخفاض أو ثابتة")

تحليل اتجاه المشاهدات - هل تتوقع ارتفاع؟

📊 متوسط المشاهدات الشهري:
   يناير (الشهر 1): 866.57 ساعة
   فبراير (الشهر 2): 823.46 ساعة
   مارس (الشهر 3): 754.65 ساعة

📈 نسبة النمو من يناير إلى مارس: -12.9%
   ❌ النتيجة: المشاهدات في انخفاض

📐 تحليل الاتجاه الخطي:
   قيمة الميل (Slope): -3.1793
   ❌ الميل سالب → المشاهدات في انخفاض

🔮 مقارنة آخر 30 يوم بالـ 30 يوم القادمة:
   آخر 30 يوم (فعلي): 695.72 ساعة
   الـ 30 يوم القادمة (متوقع): 646.78 ساعة
   ⚠️ المتوقع أقل أو مساوٍ

🏆 الإجابة النهائية:
❌ غير صحيح - المشاهدات في انخفاض أو ثابتة
